In [ ]:
import pandas as pd
import networkx as nx
import numpy as np
from collections import defaultdict
import sys

def high_salience_skeleton(table, undirected=False, return_self_loops=False):
    """
    Calculate high salience skeleton backbone (from course materials).
    
    This function identifies statistically significant edges based on shortest path
    calculations through the network.
    """
    sys.stderr.write("Calculating HSS score...\n")
    table = table.copy()
    table['distance'] = 1.0 / table['nij']
    nodes = set(table['src']) | set(table['trg'])
    G = nx.from_pandas_edgelist(table, source='src', target='trg', 
                                 edge_attr='distance', create_using=nx.DiGraph())
    cs = defaultdict(float)
    
    for s in nodes:
        pred = defaultdict(list)
        dist = {t: float('inf') for t in nodes}
        dist[s] = 0.0
        Q = defaultdict(list)
        for w in dist:
            Q[dist[w]].append(w)
        S = []
        
        while len(Q) > 0:
            v = Q[min(Q.keys())].pop(0)
            S.append(v)
            for _, w, l in G.edges(nbunch=[v,], data=True):
                new_distance = dist[v] + l['distance']
                if dist[w] > new_distance:
                    Q[dist[w]].remove(w)
                    dist[w] = new_distance
                    Q[dist[w]].append(w)
                    pred[w] = []
                if dist[w] == new_distance:
                    pred[w].append(v)
            while len(S) > 0:
                w = S.pop()
                for v in pred[w]:
                    cs[(v, w)] += 1.0
            Q = defaultdict(list, {k: v for k, v in Q.items() if len(v) > 0})
    
    table['score'] = table.apply(lambda x: cs[(x['src'], x['trg'])] / len(nodes), axis=1)
    
    if not return_self_loops:
        table = table[table['src'] != table['trg']]
    
    if undirected:
        table['edge'] = table.apply(lambda x: '%s-%s' % (min(x['src'], x['trg']), 
                                                          max(x['src'], x['trg'])), axis=1)
        table_maxscore = table.groupby(by='edge')['score'].sum().reset_index()
        table = table.merge(table_maxscore, on='edge', suffixes=('_min', ''))
        table = table.drop_duplicates(subset=['edge'])
        table = table.drop('edge', axis=1)
        table = table.drop('score_min', axis=1)
        table['score'] = table['score'] / 2.0
    
    return table[['src', 'trg', 'nij', 'score']]


def apply_backboning(df, 
                     source_col='source',
                     target_col='target', 
                     weight_col='weight',
                     min_votes_threshold=None,
                     votes_col='total_votes_shared',
                     alpha=0.0):
    """
    Apply high_salience_skeleton backboning to political voting network.
    
    Parameters:
    -----------
    df : pandas.DataFrame
        Edge list DataFrame (output from filter_and_aggregate)
    source_col : str
        Column name for source nodes
    target_col : str
        Column name for target nodes
    weight_col : str
        Column name for edge weights (agreement rate 0-1)
    min_votes_threshold : int or None
        Minimum number of shared votes required to include an edge.
        Applied BEFORE backboning. If None, no filtering is applied.
    votes_col : str
        Column name for total votes count (needed if filtering by min_votes_threshold)
    alpha : float
        Significance threshold for backbone extraction. Edges with score > alpha are kept.
        Default is 0.0 (keep all edges with any salience).
    
    Returns:
    --------
    tuple : (filtered_df, stats_dict, nx.Graph)
        - filtered_df: DataFrame with only backbone edges (preserves source_party, target_party)
        - stats_dict: Dictionary with statistics (nodes, edges before/after, etc.)
        - graph: NetworkX Graph object with backbone edges
    """
    df = df.copy()
    
    # Store original statistics
    original_edges = len(df)
    original_nodes = len(set(df[source_col]) | set(df[target_col]))
    
    # Filter by minimum votes threshold BEFORE backboning
    if min_votes_threshold is not None:
        df = df[df[votes_col] >= min_votes_threshold]
        after_filter_edges = len(df)
        after_filter_nodes = len(set(df[source_col]) | set(df[target_col]))
    else:
        after_filter_edges = original_edges
        after_filter_nodes = original_nodes
    
    # Prepare edge table for backboning (rename columns to match expected format)
    edge_table = df[[source_col, target_col, weight_col]].copy()
    edge_table.columns = ['src', 'trg', 'nij']
    
    # Apply high salience skeleton
    backbone_table = high_salience_skeleton(edge_table, undirected=True)
    
    # Apply alpha threshold
    backbone_table = backbone_table[backbone_table['score'] > alpha]
    
    # Merge back with original data to get ALL original columns
    backbone_table = backbone_table.merge(
        df, 
        left_on=['src', 'trg'], 
        right_on=[source_col, target_col],
        how='left'
    )
    
    # Clean up: keep original columns plus hss_score
    # Remove duplicate src/trg columns and the nij column from backbone
    cols_to_keep = [source_col, target_col, 'source_party', 'target_party', 
                    votes_col, 'total_votes_agreed', weight_col]
    
    # Add score as hss_score
    backbone_table['hss_score'] = backbone_table['score']
    cols_to_keep.append('hss_score')
    
    result_df = backbone_table[cols_to_keep].copy()
    
    # Create NetworkX graph with all attributes
    G = nx.from_pandas_edgelist(
        result_df, 
        source=source_col, 
        target=target_col,
        edge_attr=[weight_col, 'hss_score', votes_col, 'total_votes_agreed'],
        create_using=nx.Graph()
    )
    
    # Add party information as node attributes
    node_parties = {}
    for _, row in result_df.iterrows():
        node_parties[row[source_col]] = row['source_party']
        node_parties[row[target_col]] = row['target_party']
    
    nx.set_node_attributes(G, node_parties, 'party')
    
    # Calculate statistics
    backbone_edges = len(result_df)
    backbone_nodes = G.number_of_nodes()
    
    stats = {
        'original_nodes': original_nodes,
        'original_edges': original_edges,
        'after_min_votes_filter_nodes': after_filter_nodes,
        'after_min_votes_filter_edges': after_filter_edges,
        'backbone_nodes': backbone_nodes,
        'backbone_edges': backbone_edges,
        'nodes_retained_pct': 100.0 * backbone_nodes / original_nodes,
        'edges_retained_pct': 100.0 * backbone_edges / original_edges,
        'min_votes_threshold': min_votes_threshold,
        'alpha_threshold': alpha,
        'avg_weight': result_df[weight_col].mean(),
        'avg_hss_score': result_df['hss_score'].mean(),
    }
    
    # Print summary
    print("\n" + "="*80)
    print("BACKBONING SUMMARY")
    print("="*80)
    print(f"\nOriginal network:")
    print(f"  Nodes: {original_nodes}")
    print(f"  Edges: {original_edges}")
    
    if min_votes_threshold is not None:
        print(f"\nAfter min_votes_threshold={min_votes_threshold}:")
        print(f"  Nodes: {after_filter_nodes} ({100.0 * after_filter_nodes / original_nodes:.1f}%)")
        print(f"  Edges: {after_filter_edges} ({100.0 * after_filter_edges / original_edges:.1f}%)")
    
    print(f"\nAfter backbone extraction (alpha={alpha}):")
    print(f"  Nodes: {backbone_nodes} ({stats['nodes_retained_pct']:.1f}%)")
    print(f"  Edges: {backbone_edges} ({stats['edges_retained_pct']:.1f}%)")
    print(f"\nBackbone edge statistics:")
    print(f"  Average weight (agreement rate): {stats['avg_weight']:.3f}")
    print(f"  Average HSS score: {stats['avg_hss_score']:.4f}")
    print(f"  Weight range: [{result_df[weight_col].min():.3f}, {result_df[weight_col].max():.3f}]")
    print(f"  HSS score range: [{result_df['hss_score'].min():.4f}, {result_df['hss_score'].max():.4f}]")
    print("="*80 + "\n")
    
    return result_df, stats, G

In [ ]:
df_p66_bb, stats_p66, G_p66_bb = apply_backboning(
    df_p66,
    min_votes_threshold=10,
    alpha=0.1
)
df_p71_bb, stats_p71, G_p71_bb = apply_backboning(
    df_p71,
    min_votes_threshold=10,
    alpha=0.1
)